# Power-Law + Spline (free knots) mass model -- `tempest` run

This notebook tests `CHIMERA.population.mass.paired.pls_free_knots` (the
Power-Law + B-spline paired mass model with **inferred knot positions**)
with a `tempest` hyperlikelihood run, mirroring
`examples/bpl3p_run_tempest.py` / `examples/random_features_blocks.ipynb`
(same priors-dict / `tempest` / hyperlikelihood structure).

**Knots cannot swap, by construction.** `pls_free_knots` never samples the
knot positions directly. It samples an unconstrained `knot_logits` vector,
mapped to knot positions via `knot_logits_to_positions`: `softmax(knot_logits)`
is always non-negative and sums to 1, so its `cumsum` is monotonically
non-decreasing for **any** real-valued `knot_logits` -- there is no value a
sampler can propose that makes a later knot end up before an earlier one.
This is verified explicitly in Section 0.0 below, and is the reason the
priors below put a plain (unconstrained, unordered) `Normal` prior directly
on each `knot_logits` component rather than on the knot positions themselves
-- putting independent priors on the *positions* would need an explicit
ordering constraint (and would let a sampler propose swapped positions that
have to be rejected or sorted); parameterizing the *logits* makes swapping
structurally impossible instead of just numerically discouraged.

**Section 0** is fully local (no cluster data, no `tempest` needed) and
sanity-checks the plumbing: the no-swap guarantee itself, prior-predictive
draws of `p(m1)` with knot positions overlaid, and a jit/vmap check that
sampling `knot_logits`/`spline_coefficients` doesn't trigger per-particle
recompilation.

**Section 1** mirrors `bpl3p_run_tempest.py` for an actual `tempest`
hyperlikelihood run -- it needs the GWTC5 PE/injection files and the
`tempest` package (cluster-only), so edit `dir_data`/`dir_chain` before
running it.


In [ ]:
import os, sys
os.environ["CHIMERA_ENABLE_GPU"] = "True"
sys.path.append(os.getcwd() + '/../')

import numpy as np
import scipy.stats
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from CHIMERA.population.mass.paired import pls_free_knots
from CHIMERA.population.mass.paired.pls_free_knots import knot_logits_to_positions
from CHIMERA.population.mass import p_m1, p_m1m2


## Config

`num_inner_knots` includes the two domain endpoints (`min_knot`, `max_knot`),
so there are `num_inner_knots - 2` free interior knots and
`num_inner_knots - 1` `knot_logits` entries (softmax is over the gaps
between consecutive knots, endpoints included). `degree` is the B-spline
degree (cubic, as usual). `spline_basis_spacing='log'` matches
`knot_logits=0` to *uniform log-mass* spacing (geometric spacing in solar
masses), which is the natural default for a quantity spanning more than two
orders of magnitude.

`min_knot`/`max_knot` are kept fixed (not sampled) in this example -- the
domain the spline correction is allowed to place knots in -- while
`m_low`/`m_high` (the hard truncation edges) are still sampled, same as in
the other `tempest` notebooks.

In [ ]:
num_inner_knots = 5          # includes the 2 endpoints -> 3 free interior knots
degree = 3                   # cubic B-spline
spline_basis_spacing = 'log'
spline_grid_res = 1000
min_knot, max_knot = 1.0, 150.0   # fixed spline-placement domain (not sampled)

num_coeffs = num_inner_knots + degree - 1   # see pls_free_knots.py
num_knot_logits = num_inner_knots - 1

knot_logit_prior_scale = 1.5   # moderate deviation from uniform spacing; softmax saturates for |logit| >> this
spline_c_prior_scale = 1.5     # mild log-density correction (B-spline rows sum to 1, so this upper-bounds std(spline(m)))

# tempest / hyperlikelihood run settings (Section 1) -- modest defaults for a *test* run;
# bump these up for a production fit, same knobs as bpl3p_run_tempest.py's argparse flags.
n_particles = 256
n_steps = 5
n_total = 4096
prng_key = 0
resample = 'syst'

dir_data = '/leonardo_work/IscrC_MLGW/gwtc5/data/'
dir_chain = '/leonardo_work/IscrC_MLGW/gwtc5/res/tempest/'
chain_label = 'gwtc5_pls_free_knots_tempest'

print(f"num_inner_knots={num_inner_knots} -> {num_inner_knots - 2} free interior knots, "
      f"{num_knot_logits} knot_logits, {num_coeffs} spline coefficients")


## Section 0.0 -- the no-swap guarantee, checked directly

Draw `knot_logits` from a much wider range than the prior will ever propose
(including large-magnitude values that push `softmax` deep into saturation),
map them through `knot_logits_to_positions`, and check that the resulting
knot array is never *reordered*. This is a structural property of `softmax`
+ `cumsum`: `softmax` outputs are always non-negative, so `cumsum` only ever
adds non-negative increments and is therefore monotonically non-decreasing,
in exact arithmetic *and* under float32 rounding -- there is no value a
sampler can propose that makes a later knot end up before an earlier one.

(At truly extreme `|knot_logits|`, `softmax` can saturate hard enough that
the very smallest gaps underflow to exactly `0.0` in float32, so two knots
can end up *tied* rather than swapped -- this is checked and shown below
too, and only ever happens far outside the prior's actual range.)


In [ ]:
rng = np.random.default_rng(0)
n_swap_checks = 20000

# Deliberately much wider than knot_logit_prior_scale, including extreme values.
extreme_logits = rng.normal(loc=0.0, scale=10.0, size=(n_swap_checks, num_knot_logits))
extreme_logits[:5] = rng.uniform(-50.0, 50.0, size=(5, num_knot_logits))  # a few very extreme draws

def decode(logits_batch):
    return jax.vmap(
        lambda kl: knot_logits_to_positions(kl, min_knot, max_knot, spacing=spline_basis_spacing)
    )(jnp.asarray(logits_batch))

knots_batch = decode(extreme_logits)
diffs = jnp.diff(knots_batch, axis=1)

# The guarantee that actually matters -- knots can never be REORDERED (a strict
# decrease), for any knot_logits: softmax outputs are >= 0 and cumsum only ever adds
# non-negative numbers, so it is monotonically non-decreasing even under float32
# rounding. This holds at every scale tested below, including whopping |knot_logits| ~ 50.
n_real_swaps = int(jnp.sum(jnp.any(diffs < 0, axis=1)))
print(f"{n_swap_checks} random (including extreme) knot_logits draws -> "
      f"knot arrays with a REAL swap (a decrease): {n_real_swaps} (must be 0)")
assert n_real_swaps == 0, "knot ordering guarantee violated -- should be impossible"

# At extreme |knot_logits| (>~15-20), softmax saturates in float32 and the *smallest*
# gaps can underflow to exactly 0.0 -- two knots then tie rather than swap. This is a
# floating-point artifact of stress-testing far outside anything the prior below will
# ever propose, not a violation of the guarantee (order is preserved, just degenerate).
n_ties = int(jnp.sum(jnp.any(diffs == 0, axis=1)))
print(f"knot arrays with a tie (float32 underflow at extreme |knot_logits|): {n_ties}")

# At the actual prior width, gaps stay comfortably resolved (strictly positive, no ties).
realistic_logits = rng.normal(loc=0.0, scale=knot_logit_prior_scale, size=(n_swap_checks, num_knot_logits))
realistic_diffs = jnp.diff(decode(realistic_logits), axis=1)
n_realistic_bad = int(jnp.sum(jnp.any(realistic_diffs <= 0, axis=1)))
print(f"at the actual prior scale ({knot_logit_prior_scale}): "
      f"non-strictly-increasing draws = {n_realistic_bad} / {n_swap_checks} (must be 0)")
assert n_realistic_bad == 0

fig, ax = plt.subplots(1, 1, figsize=(7, 2.5))
for i in range(min(200, n_swap_checks)):
    ax.plot(np.asarray(knots_batch[i]), np.zeros(num_inner_knots) + i, '.', color='C0', alpha=0.15, ms=3)
ax.set_xscale('log')
ax.set_xlabel('knot position (never reordered, by construction)')
ax.set_yticks([])
ax.set_title(f'{min(200, n_swap_checks)} extreme knot_logits draws -- knots never cross')
fig.tight_layout()
plt.show()


## Priors

Same style as `bpl3p_run_tempest.py`/`random_features_blocks.ipynb`:
`scipy.stats`-based `uniform`/`loguniform`/`norm` helpers, one prior per
scalar hyperparameter. `knot_logit{i}` (one per free logit) and
`spline_c{i}` (one per B-spline coefficient) are later re-assembled into the
`knot_logits`/`spline_coefficients` arrays that `pls_free_knots.update()`
expects -- exactly as `w_out{i}`/`scale{i}` are for `random_features_density`.

Putting an ordinary, unordered `Normal` prior on each `knot_logit{i}`
independently is safe *only* because of the guarantee checked above: no
matter what `tempest` proposes for these logits, the decoded knot positions
stay strictly ordered.

In [ ]:
def uniform(low, high):
  return scipy.stats.uniform(loc=low, scale=high - low)

def loguniform(low, high):
  return scipy.stats.loguniform(a=low, b=high)

def norm(loc, scale):
  return scipy.stats.norm(loc=loc, scale=scale)

# --- mass-model (pls_free_knots) priors ---
mass_priors = {
  "alpha": uniform(-4., 12.),              # power-law spectral index
  "beta": uniform(-4., 12.),               # mass-ratio pairing power law
  "delta_m": loguniform(0.01, 10.),        # low-mass smoothing scale
  "m_low": uniform(0.4, 1.4),
  "m_high": uniform(50., 200.),
}

# unconstrained knot-spacing logits -- see the "no-swap guarantee" section above
knot_logit_priors = {f"knot_logit{i}": norm(loc=0., scale=knot_logit_prior_scale) for i in range(num_knot_logits)}

# shared Normal prior on every B-spline coefficient (mild log-density correction)
spline_c_priors = {f"spline_c{i}": norm(loc=0., scale=spline_c_prior_scale) for i in range(num_coeffs)}

# --- cosmology / rate priors (same as bpl3p_run_tempest.py) ---
cosmo_rate_priors = {
  "H0": uniform(10., 200.),
  "gamma": uniform(0., 12.),
  "kappa": uniform(0., 6.),
  "zp": uniform(0., 4.),
}

priors = {**cosmo_rate_priors, **mass_priors, **knot_logit_priors, **spline_c_priors}
params_keys = list(priors.keys())
n_dim = len(params_keys)
prior_dists = [priors[k] for k in params_keys]

def prior_transform(u):
  return np.array([dist.ppf(u[i]) for i, dist in enumerate(prior_dists)])

print(f"n_dim = {n_dim}  ({num_knot_logits} knot_logits + {num_coeffs} spline_c "
      f"+ {len(mass_priors)} mass + {len(cosmo_rate_priors)} cosmo/rate)")


## Section 0.1 -- prior-predictive draws, with knot positions overlaid

Draw `knot_logits`/`spline_coefficients` (and the other mass hyperparameters)
straight from the priors above, build a `pls_free_knots` realization for
each draw, and overlay `p(m1)` with its decoded knot positions marked on the
curve -- confirming visually, not just via the assertion above, that knots
never cross as the spline shape varies across the prior.

In [ ]:
n_draws = 12
m_low_fixed, m_high_fixed = 0.92, 124.2  # fixed mass range for this prior-predictive check

base_model = pls_free_knots(
  num_inner_knots=num_inner_knots, degree=degree, spline_basis_spacing=spline_basis_spacing,
  spline_grid_res=spline_grid_res, min_knot=min_knot, max_knot=max_knot,
  m_low=m_low_fixed, m_high=m_high_fixed,
)

rng = np.random.default_rng(1)
m1 = jnp.linspace(m_low_fixed * 1.01, m_high_fixed * 0.99, 4000)

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
drawn_knots = []
for i in range(n_draws):
  knot_logits = jnp.array([knot_logit_priors[f"knot_logit{j}"].rvs(random_state=rng) for j in range(num_knot_logits)])
  spline_coefficients = jnp.array([spline_c_priors[f"spline_c{j}"].rvs(random_state=rng) for j in range(num_coeffs)])
  alpha = mass_priors["alpha"].rvs(random_state=rng)
  beta = mass_priors["beta"].rvs(random_state=rng)
  delta_m = mass_priors["delta_m"].rvs(random_state=rng)

  model_i = base_model.update(
    knot_logits=knot_logits, spline_coefficients=spline_coefficients,
    alpha=alpha, beta=beta, delta_m=delta_m,
  )
  knots_i = knot_logits_to_positions(knot_logits, min_knot, max_knot, spacing=spline_basis_spacing)
  drawn_knots.append(np.asarray(knots_i))

  line, = ax.plot(m1, p_m1(model_i, m1), alpha=0.7, lw=1.2)
  p_at_knots = p_m1(model_i, jnp.clip(knots_i, m_low_fixed * 1.01, m_high_fixed * 0.99))
  ax.plot(knots_i, p_at_knots, 'o', color=line.get_color(), ms=4, alpha=0.8)

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('$m_1$'); ax.set_ylabel('$p(m_1)$')
ax.set_title(f'{n_draws} prior-predictive draws (dots = decoded knot positions)')
fig.tight_layout()
plt.show()

drawn_knots = np.array(drawn_knots)
n_bad = int(np.sum(np.any(np.diff(drawn_knots, axis=1) <= 0, axis=1)))
print(f"knot arrays that are NOT strictly increasing across {n_draws} draws: {n_bad} (must be 0)")


## Section 0.2 -- knot-position spread under the prior

Knot slot `0` and slot `num_inner_knots - 1` are always exactly `min_knot`
and `max_knot` (the softmax's cumulative sum starts at 0 and ends at
exactly 1 by construction), so only the `num_inner_knots - 2` *interior*
knot slots actually move under the prior. Histogram the decoded position of
each interior slot across many prior draws of `knot_logits`. The individual
distributions can overlap freely between neighbouring knots (a wide
`knot_logit_prior_scale` lets knot `i` occasionally sit further out than a
particular draw of knot `i+1` would under a *different* draw) -- what
matters, and what is checked explicitly, is that **within a single draw**
the knots are always sorted.

In [ ]:
n_hist = 4000
logit_draws = np.stack([
  [knot_logit_priors[f"knot_logit{j}"].rvs(random_state=rng) for j in range(num_knot_logits)]
  for _ in range(n_hist)
])
knots_draws = jax.vmap(
    lambda kl: knot_logits_to_positions(kl, min_knot, max_knot, spacing=spline_basis_spacing)
)(jnp.asarray(logit_draws))
knots_draws = np.asarray(knots_draws)

assert np.all(np.diff(knots_draws, axis=1) > 0), "knot ordering guarantee violated -- should be impossible"
assert np.allclose(knots_draws[:, 0], min_knot) and np.allclose(knots_draws[:, -1], max_knot), \
    "endpoint knots should be exactly fixed at min_knot/max_knot"

interior = range(1, num_inner_knots - 1)  # slots 0 and -1 are fixed at min_knot/max_knot

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
for k in interior:
  ax.hist(knots_draws[:, k], bins=60, alpha=0.5, label=f'knot {k}', color=f"C{k % 10}")
ax.set_xscale('log')
ax.set_xlabel('knot position')
ax.set_ylabel('count')
ax.set_title(f'{n_hist} prior draws -- interior-knot-slot position spread (still sorted within every draw)')
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
plt.show()


## Section 0.3 -- jit/vmap check: one compile, many `knot_logits`/`spline_coefficients` values

Batch several particles with *different* `knot_logits`/`spline_coefficients`
through `.update()` + `p_m1` under a single `jax.jit` + `jax.vmap`, exactly
the way `tempest` will evaluate a particle population. The second call
reuses the trace (no recompilation) even though its `knot_logits` values
differ from the first call's.

In [ ]:
import time

n_particles_test = 64
logits_batch = jnp.stack([
  jnp.array([knot_logit_priors[f"knot_logit{j}"].rvs(random_state=rng) for j in range(num_knot_logits)])
  for _ in range(n_particles_test)
])
coeffs_batch = jnp.stack([
  jnp.array([spline_c_priors[f"spline_c{j}"].rvs(random_state=rng) for j in range(num_coeffs)])
  for _ in range(n_particles_test)
])

@jax.jit
def eval_particle(knot_logits, spline_coefficients):
  model_i = base_model.update(knot_logits=knot_logits, spline_coefficients=spline_coefficients)
  return p_m1(model_i, m1)

t0 = time.time()
out1 = jax.vmap(eval_particle)(logits_batch, coeffs_batch)
out1.block_until_ready()
t1 = time.time()

logits_batch_2 = logits_batch * 1.3 + 0.5  # different concrete values, same shapes
out2 = jax.vmap(eval_particle)(logits_batch_2, coeffs_batch)
out2.block_until_ready()
t2 = time.time()

print(f"first call (compile + run): {t1 - t0:.3f}s")
print(f"second call, different knot_logits values (should be much faster -- no retrace): {t2 - t1:.3f}s")
assert not jnp.any(jnp.isnan(out1)) and not jnp.any(jnp.isnan(out2))
print("no NaNs -- vmap over distinct knot_logits works with a single jit trace.")


## Section 1 -- full `tempest` hyperlikelihood run (cluster only)

Mirrors `examples/bpl3p_run_tempest.py` / `examples/random_features_blocks.ipynb`
exactly in structure: load GWTC5 PE + injection data, build
`cosmo`/`mass`/`rate` -> `population` -> `hyperlikelihood`, define
`log_likelihood`, run `tempest.Sampler`, save posterior + blobs to
`arviz.InferenceData`.

Needs `dir_data`/`dir_chain` (set in the config cell above) to point at real
GWTC5 files, and the `tempest` package (both cluster-only) -- **edit the
paths before running this section**.

In [ ]:
from CHIMERA import data
from CHIMERA.cosmo import flrw
from CHIMERA.rate import madau_dickinson
from CHIMERA import population as population_module, selection_function, hyperlikelihood
import h5py
import tempest as tp

## Load PE data
file_ev = dir_data + 'GWTC5_242CBC_FAR0.25_PE2048.h5'
pe_gw = data.load_gw_pe_samples(file_ev, parameters=['m1det', 'm2det', 'dL'], return_struct=True)
pe_prior = jnp.load(dir_data + 'GWTC5_242CBC_FAR0.25_PE2048prior.npy')
pe_gw = pe_gw.update(pe_prior=pe_prior)
Nev = len(pe_gw.dL)

## Load injection data and define selection function
file_inj = dir_data + "GWTC5_injections_far0.25.h5"
f_inj = h5py.File(file_inj)
Ngen = float(f_inj.attrs['Ngen'])
Tobs = float(f_inj.attrs['Tobs'])
theta_inj_det = data.load_injection_data(file_inj, snr_cut=None, key_mapping={'snr': 'snr', 'log_pdraw': 'log_pdraw'}, frame='detector')

sel_fcn = selection_function(theta_inj_det, N_inj=Ngen)


In [ ]:
cosmo = flrw(H0=67.9, Om0=0.3065, z_max=5.)
mass = pls_free_knots(
  num_inner_knots=num_inner_knots, degree=degree, spline_basis_spacing=spline_basis_spacing,
  spline_grid_res=spline_grid_res, min_knot=min_knot, max_knot=max_knot,
  m_low=0.9, m_high=124.,
)
rate = madau_dickinson()
population = population_module(cosmo, mass, rate, scale_free=True, Tobs=Tobs, R0=1)

hyperlike = hyperlikelihood(
  theta_gw_det=pe_gw,
  population=population,
  z_grids_res=300,
  selection_function=sel_fcn,
  pe_neff=2.0,
  inj_neff=None,
  kind_kde='fft',
  kernel='epan',
  kde_bw=None,
)


Assembling `hyperparams` from the flat `tempest` particle vector:
`dict(zip(params_keys, params))` gives every scalar hyperparameter directly;
`knot_logit{i}`/`spline_c{i}` are popped out and re-stacked into the
`knot_logits`/`spline_coefficients` arrays that `pls_free_knots.update()`
(via `mass.keys`) expects -- exactly the same pattern
`random_features_blocks.ipynb` uses for `w_out{i}`/`scale{i}`.

In [ ]:
def log_likelihood(params):
  raw = dict(zip(params_keys, params))
  knot_logits = jnp.array([raw.pop(f"knot_logit{i}") for i in range(num_knot_logits)])
  spline_coefficients = jnp.array([raw.pop(f"spline_c{i}") for i in range(num_coeffs)])
  hyperparams = {**raw, "knot_logits": knot_logits, "spline_coefficients": spline_coefficients}
  log_like_evs, N_exp, neff_inj, log_like = hyperlike.compute_all(**hyperparams)
  return log_like, log_like_evs, N_exp, neff_inj

# Prior-predictive finite-likelihood check before launching the full sampler: with n_particles
# particles drawn from the prior, tempest needs enough of them finite to seed its initial
# population from. Keep the accepted draws around too, to inspect what they look like as p(m1).
n_check = 10000
accepted_thetas = []
for s in range(n_check):
  theta = prior_transform(np.random.default_rng(s).uniform(size=n_dim))
  ll, *_ = log_likelihood(theta)
  if np.isfinite(ll):
    accepted_thetas.append(theta)
print(f"{len(accepted_thetas)}/{n_check} prior draws gave a finite log-likelihood "
      f"(~{len(accepted_thetas)/n_check*n_particles:.0f} expected finite particles at init)")


### What does the finite-likelihood region of the prior look like as a mass function?

Before spending compute on the full run, plot `p(m1)` for a handful of the
`accepted_thetas` collected above, with decoded knot positions marked --
these are draws that passed every event's `pe_neff` check, so this is a
sanity check on what the sampler's *initial* population actually looks
like, not yet the posterior.

In [ ]:
n_plot = min(20, len(accepted_thetas))
plot_idx = np.random.default_rng(0).choice(len(accepted_thetas), size=n_plot, replace=False)
m1_grid = jnp.linspace(0.5, 200., 3000)

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
for i in plot_idx:
  raw = dict(zip(params_keys, accepted_thetas[i]))
  knot_logits_i = jnp.array([raw.pop(f"knot_logit{k}") for k in range(num_knot_logits)])
  spline_c_i = jnp.array([raw.pop(f"spline_c{k}") for k in range(num_coeffs)])
  mass_i = mass.update(
    knot_logits=knot_logits_i, spline_coefficients=spline_c_i,
    alpha=raw["alpha"], beta=raw["beta"], delta_m=raw["delta_m"],
    m_low=raw["m_low"], m_high=raw["m_high"],
  )
  knots_i = knot_logits_to_positions(knot_logits_i, min_knot, max_knot, spacing=spline_basis_spacing)
  line, = ax.plot(m1_grid, p_m1(mass_i, m1_grid), alpha=0.6, lw=1)
  p_at_knots = p_m1(mass_i, jnp.clip(knots_i, m1_grid[0], m1_grid[-1]))
  ax.plot(knots_i, p_at_knots, 'o', color=line.get_color(), ms=3, alpha=0.6)

ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("$m_1$"); ax.set_ylabel("$p(m_1)$")
ax.set_title(f"{n_plot} accepted (finite-likelihood) prior draws")
fig.tight_layout()
plt.show()


In [ ]:
sampler = tp.Sampler(
  prior_transform=prior_transform,
  log_likelihood=log_likelihood,
  n_dim=n_dim,
  n_particles=n_particles,
  n_steps=n_steps,
  resample=resample,
  random_state=prng_key,
  output_dir=dir_chain,
  output_label=chain_label,
)

sampler.run(n_total=n_total, save_every=1, resume_state_path=None)

samples, weights, logl, blobs = sampler.posterior(resample=True, return_blobs=True)
samples_dict = {k: samples[:, i] for i, k in enumerate(priors)}

log_like_evs_blob = blobs[:, 0]
N_exp_blob = blobs[:, 1]
neff_inj_blob = blobs[:, 2]

logZ, logZerr = sampler.evidence()
print(f"logZ = {logZ}  logZerr = {logZerr}")


In [ ]:
import xarray as xr
import arviz as az

n_chains = 1
n_draws_post = len(samples)
n_events = np.stack(log_like_evs_blob).shape[-1]

posterior_ds = xr.Dataset(
  {k: (['chain', 'draw'], v.reshape(n_chains, n_draws_post)) for k, v in samples_dict.items()}
)
sample_stats_ds = xr.Dataset({
  'log_likelihood': (['chain', 'draw'], logl.reshape(n_chains, n_draws_post)),
  'weights': (['chain', 'draw'], weights.reshape(n_chains, n_draws_post)),
  'log_like_evs': (['chain', 'draw', 'event'], np.stack(log_like_evs_blob).reshape(n_chains, n_draws_post, n_events)),
  'N_exp': (['chain', 'draw'], np.asarray(N_exp_blob).reshape(n_chains, n_draws_post)),
  'Neff_inj': (['chain', 'draw'], np.asarray(neff_inj_blob).reshape(n_chains, n_draws_post)),
})

idata = az.InferenceData(posterior=posterior_ds, sample_stats=sample_stats_ds)
idata.attrs['log_evidence'] = float(logZ)
idata.attrs['log_evidence_err'] = str(logZerr) if logZerr is None else float(logZerr)
idata.to_netcdf(f"{dir_chain}/{chain_label}_final.nc")


## Section 2 -- inspecting the posterior over knot positions

Decodes each posterior draw's `knot_logit{i}` into physical knot positions
and re-checks the ordering guarantee on *posterior* (not just prior) draws,
then compares the posterior spread of each knot slot against its prior.
Also draws a posterior-predictive `p(m1)` band.

In [ ]:
knot_logits_post = np.stack([samples_dict[f"knot_logit{i}"] for i in range(num_knot_logits)], axis=-1)
knots_post = np.asarray(jax.vmap(
    lambda kl: knot_logits_to_positions(kl, min_knot, max_knot, spacing=spline_basis_spacing)
)(jnp.asarray(knot_logits_post)))

n_bad_post = int(np.sum(np.any(np.diff(knots_post, axis=1) <= 0, axis=1)))
print(f"posterior knot arrays that are NOT strictly increasing: {n_bad_post} / {len(knots_post)} (must be 0)")

# Slots 0 and -1 are fixed at min_knot/max_knot (see Section 0.2) -- only the interior
# slots have anything to plot.
interior = list(range(1, num_inner_knots - 1))

fig, axes = plt.subplots(1, len(interior), figsize=(3.2 * len(interior), 3.2), sharey=False)
prior_logits_draws = np.stack([
  [knot_logit_priors[f"knot_logit{j}"].rvs(random_state=np.random.default_rng(100 + j)) for j in range(num_knot_logits)]
  for _ in range(5000)
])
prior_knots_draws = np.asarray(jax.vmap(
    lambda kl: knot_logits_to_positions(kl, min_knot, max_knot, spacing=spline_basis_spacing)
)(jnp.asarray(prior_logits_draws)))

for ax, k in zip(np.atleast_1d(axes), interior):
  ax.hist(prior_knots_draws[:, k], bins=40, density=True, alpha=0.4, label='prior', color='gray')
  ax.hist(knots_post[:, k], bins=40, density=True, alpha=0.7, label='posterior', color=f"C{k % 10}")
  ax.set_xscale('log')
  ax.set_title(f"knot {k}")
  ax.legend(fontsize=7)
fig.tight_layout()
plt.show()


In [ ]:
n_ppc = 200
idx = np.random.default_rng(0).choice(len(samples), size=n_ppc, replace=False)
m1_grid = jnp.linspace(0.5, 150., 3000)

curves = []
for i in idx:
  knot_logits_i = jnp.array([samples_dict[f"knot_logit{k}"][i] for k in range(num_knot_logits)])
  spline_c_i = jnp.array([samples_dict[f"spline_c{k}"][i] for k in range(num_coeffs)])
  model_i = mass.update(
    knot_logits=knot_logits_i, spline_coefficients=spline_c_i,
    alpha=samples_dict['alpha'][i], beta=samples_dict['beta'][i], delta_m=samples_dict['delta_m'][i],
    m_low=samples_dict['m_low'][i], m_high=samples_dict['m_high'][i],
  )
  curves.append(p_m1(model_i, m1_grid))

curves = jnp.stack(curves)
lo, med, hi = jnp.percentile(curves, jnp.array([5., 50., 95.]), axis=0)

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.fill_between(m1_grid, lo, hi, alpha=0.3, label='90% posterior-predictive')
ax.plot(m1_grid, med, lw=1.5, label='median')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('$m_1$'); ax.set_ylabel('$p(m_1)$')
ax.legend()
fig.tight_layout()
plt.show()
